RAG based model which takes a Web URL as its Knowledge Base and generates response to the queries of the Learners

In [ ]:
!pip install langchain langchain-core langchain_community langgraph langchain-huggingface transformers torch

In [ ]:
!pip install unstructured
from langchain_community.document_loaders import UnstructuredURLLoader

In [ ]:
url=['https://langchain-ai.github.io/langgraph/tutorials/introduction/']
loader=UnstructuredURLLoader(urls=url)
docs=loader.load()

In [ ]:
print(docs)

[Document(metadata={'source': 'https://langchain-ai.github.io/langgraph/tutorials/introduction/'}, page_content='Redirecting...')]


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits=text_splitter.split_documents(docs)

In [ ]:
from langchain_core import embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings()

/tmp/ipykernel_20287/2191471436.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings()
/tmp/ipykernel_20287/2191471436.py:3: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings=HuggingFaceEmbeddings()
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/toke

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
!pip install langchain_chroma

In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document

vectorstore=Chroma.from_documents(documents=all_splits,embedding=HuggingFaceEmbeddings())

/tmp/ipykernel_20287/2723607677.py:4: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  vectorstore=Chroma.from_documents(documents=all_splits,embedding=HuggingFaceEmbeddings())


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate #input variables and the template they help us in instructing the LLM in a structured manner
from transformers import pipeline #pre-trained model from HF, this pipeline will already know alot of things
from langchain_core.output_parsers import StrOutputParser
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
model_id="tiiuae/falcon-7b"

In [ ]:
text_generation_pipeline=pipeline("text-generation",
                                  model=model_id,
                                  model_kwargs={'torch_dtype':torch.bfloat16},
                                  max_new_tokens=200,
                                  device=0,
                                  temperature=0.7, #if the value is low,
                                  #it means the response will be more deterministic and if the value is high, the response is random and creative
)

llm=HuggingFacePipeline(pipeline=text_generation_pipeline)

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
!pip install langchainhub

In [ ]:
!pip install langchain --upgrade

In [ ]:
from langchain_classic import hub

In [ ]:
from langchain_core.prompts import PromptTemplate

template="""Use the following piece of context to answer the question.
#If you don't know the answer, just say that you don't know, don't try to make up an answer.
#Use three sentences maximum to answer the question and try to keep the answer as concise and to-the-point as possible
#Always say "Thanks for asking!" at the end of the answer.
{context}
Question:{question}
Helpful Answer:"""

prompt=PromptTemplate.from_template(template)

prompt=hub.pull("rlm/rag-prompt") # Call pull directly from the module

In [ ]:
from typing_extensions import List, TypedDict #is a special type of Dict where we also specify the dtype along with the key-value pair
class State(TypedDict):
  question:str
  context:List[Document]
  answer:str

In [ ]:
def retrieve(state:State):
  retrieved_docs=vectorstore.similarity_search(state['question'],k=1)
  return {'context':retrieved_docs}

In [ ]:
def generate(state:State):
  docs_content="\n\n".join(doc.page_content for doc in state['context'])
  messages=prompt.invoke({'question':state['question'],'context':docs_content})
  response=llm.invoke(messages)
  return {'answer':response}


In [ ]:
initial_state = {'question': "What is LangGraph?", 'context': []}
final_state = generate(initial_state)
print(final_state['answer'])

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
